<!-- <a href="https://colab.research.google.com/github/prane-eth/iRAT/blob/main/Result-filter/Result-filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> -->

## Result filter module (_Attention-Retrieval_)

In [1]:
import sys
if 'google.colab' in sys.modules:
	!pip install --upgrade datasets
	from IPython.display import clear_output
	clear_output()

import os
if os.path.exists('data'):
	os.chdir('data')
if not os.getcwd().endswith('data'):
	raise RuntimeError('Current directory is not "data".')

In [2]:
import os
if not os.path.exists('coding_dataset'):
	# print('Creating coding_dataset...')
	# os.system('python 1_get_coding_rows.py')
	# os.system('python 2_create_dataset.py')
	raise FileNotFoundError('The dataset directory "coding_dataset" does not exist. Please run the dataset creation scripts first.')

from datasets import load_from_disk

# restores the same DatasetDict with train/validation splits
dataset = load_from_disk('coding_dataset')
if not len(dataset['train']) or not len(dataset['validation']):
	raise ValueError('The training dataset is empty. Please check the dataset creation process.')

print('Sample rows:')
for i, row in enumerate(dataset['train']):
	if i > 1:  # Display only the first 5 rows
		break
	print(f'Row {i}')
	print(f'  Query: {row["query"]}')

Sample rows:
Row 0
  Query: what is an of clause sql
Row 1
  Query: javascript define array


In [ ]:
# Load the LLM
from dotenv import load_dotenv

load_dotenv(override=True)

def user_message(query: str):
	return {'role': 'user', 'content': query}

def system_message(query: str):
	return {'role': 'system', 'content': query}


# # Load the LLM
# model_name = 'microsoft/Phi-4-mini-instruct'
# model_name_short = 'phi4m'

# # from transformers import pipeline
# # model = pipeline('question-answering', model=model_name)

# # def get_response(messages: list[str]) -> str:
# # 	if not isinstance(messages, list):
# # 		raise ValueError('Input should be a list of messages')
# # 	response = model(messages)
# # 	return response['answer']  # [0]['answer']


# # Error for above code due to insufficient memory.
# # Use GitHub API

# from azure.ai.inference import ChatCompletionsClient
# from azure.ai.inference.models import user_message
# from azure.core.credentials import AzureKeyCredential
# load_dotenv('.env.main')  # my local file

# endpoint = 'https://models.github.ai/inference'
# github_token = os.getenv('GITHUB_MODELS_TOKEN')

# openai_client = ChatCompletionsClient(endpoint=endpoint, credential=AzureKeyCredential(github_token))

# def predict(query: str) -> str:
# 	response = openai_client.complete(
# 		messages=[
# 			user_message(query),
# 		],
# 		model=model_name,
# 		temperature=0.3,
# 	)
# 	response = response.choices[0].message.content
# 	return response


# Using Llama 3.3 through Groq API
import openai

model_name = 'llama-3.3-70b-versatile'
model_name_short = 'llama-3.3'

def predict(messages) -> str:
	response = openai.chat.completions.create(
		model=model_name,
		messages=messages,
		temperature=0.1,
		max_tokens=512,
	)
	response = response.choices[0].message.content
	return response

In [4]:
# store in a file
import json
results_filename = f'filter-eval_results-{model_name_short}.json'
def save_results(results):
	with open(results_filename, 'w') as f:
		json.dump(results, f, indent=4)

def load_results():
	try:
		with open(results_filename, 'r') as f:
			results = json.load(f)
		return results
	except FileNotFoundError:
		save_results({})
		return {}

results = load_results()

def mark_as_correct(index, score=True):
	results[str(index)] = score
	# print(f'Correct')
	save_results(results)

def mark_as_incorrect(index, comment=None):
	results[str(index)] = False
	print(f'Incorrect')
	if comment:
		print('\t -', comment)
	save_results(results)

def get_response(messages) -> str:
	for attempt in range(3):
		try:
			response = predict(messages)
			if not response:
				raise ValueError('Empty response from the model')

			with open(f'response.log', 'a') as f:
				print(response, file=f)
				print('---------------------', file=f)

			return response
		except KeyboardInterrupt as e:
			print('Interrupted by user')
			raise e
		except Exception as e:
			pass
	if 'e' in locals():
		print(f'Error: {e}')
		raise e
	raise RuntimeError(f'Failed to get a valid response after {attempt+1} attempts')

results

{}

In [ ]:
with open('response.log', 'w') as f:
	f.write('')

def extract_selected_urls(response, urls):
	selected_urls = []
	for url in urls:
		if f'{url}: 2' in response:
			selected_urls.append(url)
		elif f'{url}: 1' in response:
			selected_urls.append(url)
		elif f'{url}: 0' in response:
			continue
		else:
			print(f'Error for {url}: answer not found or not a digit')
			raise ValueError(f'Answer not found for {url}')
	return selected_urls

system_msg = (
	'Act as an expert in evaluating the relevance of passages to a given query.\n'
	'Your task is to determine whether the provided passages answer the query based on their content, not the URLs.\n'
	'You will be given a query and a list of passages with their URLs.\n'
)

def get_response_urls(query, urls, passage_texts):
	messages = [
		system_message(system_msg),
	]
	# ask the model whether the query is answered by each passage text
	prompt = ''
	for url, passage_text in zip(urls, passage_texts):
		prompt += f'URL: {url}\nPassage: {passage_text} --- \n'
	prompt += (
		f'Given the query: "{query}"\n'
		'For each passage above, rate its relevance to the query using ONLY the passage content (not the URL):\n'
		'- 2: Best match (most relevant to the query without spammy text)\n'
		'- 1: Good match (partially answers the query)\n'
		'- 0: Not relevant or spam/unrelated\n'
		'Do NOT skip any URL. Only rate based on passage content.\n'
		'Respond STRICTLY in this format:\n'
  			'Thoughts:\nhttps://example.com: This URL is about ....\nhttps://example2.com: This URL is not about ...\n'
			'Response:\nhttps://example.com: 1\nhttps://example2.com: 0\n'
		'First mention each URL and write your THOUGHTS followed by a response in the specified format.\n'
		'Vote 1 only if the passage is ACTUALLY RELEVANT to answer the query, not just because it contains some keywords.\n'
		'Feel free to vote 0 for all if required.\n'
	)
	messages.append(user_message(prompt))
	response = get_response(messages)
	'''# Common mistake: Thoughts or Response not mentioned
	if 'Thoughts:' not in response or 'Response:' not in response:
		print('Mistake - Thoughts or Response not mentioned. Retrying...')
		try:
			response = get_response(messages)
		except Exception as e:
			print('Mistake again - Thoughts or Response still not mentioned. Please check.')
			raise ValueError('Thoughts or Response not mentioned in the response')'''
	# Common mistake: Some URLs are not mentioned
	try:
		response = extract_selected_urls(response, urls)
		# Now, the response is a list of selected URLs
		return response
	except ValueError as e:
		print('Mistake - Some URLs missing. Retrying...')
		try:
			response = get_response(messages)
			return response
		except Exception as e:
			print('Mistake again - Still some URLs missing. Please check.')
			raise ValueError('Some URLs are missing in the response')

# https://huggingface.co/datasets/microsoft/ms_marco/viewer/v2.1/validation
def evaluate_selections(selected_indices, correct_indices, val_index):
	if not correct_indices:
		# no need to select anything, so consider it correct
		mark_as_correct(val_index)
		return
	total_correct = len(correct_indices)
	correct_selected = len([index for index in selected_indices if index in correct_indices])
	score = correct_selected / total_correct
	if score:
		mark_as_correct(val_index, score)
	else:
		mark_as_incorrect(val_index, 'None of the correct indices were selected')

for val_index, row in enumerate(dataset['validation']):
	if results.get(str(val_index)) in [True, False]:  # already evaluated
		continue

	print('---', val_index)
	correct_indices = [i for i, x in enumerate(row['passages']['is_selected']) if x > 0]
	urls = row['passages']['url']
	try:
		selected_urls = get_response_urls(row['query'], urls, row['passages']['passage_text'])
	except Exception as e:
		print(f'Error processing row {val_index}: {e}')
		# mark_as_incorrect(index, 'Error during processing')
		continue

	# Extract the selected rating for each URL
	selected_indices = []
	for url_index, url in enumerate(urls):
		if url in selected_urls:
			selected_indices.append(url_index)

	evaluate_selections(selected_indices, correct_indices, val_index)

total_evaluated = len(results)
total_correct = sum(results.values())
print(f'Total evaluated: {total_evaluated}, Total correct: {total_correct}')
accuracy = total_correct / total_evaluated if total_evaluated > 0 else 0
print(f'Accuracy: {accuracy:.2%}')
print(f'Model: {model_name}')

with open('scores_LLM-1.txt', 'a') as f:
	f.write(f'{model_name_short} - accuracy: {accuracy:.2%}\n')